# Let's try some **regression**!!

## Simple Regression

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import src.machine_learning as ML


class GlyphClassifier(nn.Module):
    def __init__(self, resolution):
        super(GlyphClassifier, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 * resolution[0]//8 * resolution[1]//8, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1)
        )
 
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)  # Flatten the output
        x = self.classifier(x)
        return x

In [ ]:
# Getting the dataset
dataset_file = 'data/random-stars-L.zip'
train_dataset = ML.GlyphDataset(dataset_file, resize=(128,128), split = "train",mode="regression")
test_dataset = ML.GlyphDataset(dataset_file, resize=(128,128),split = 'test',mode='regression')
train_dataset.show()

# Assign the loaders 

train_loader = ML.create_loader(train_dataset, batch_size=64, shuffle = True)
test_loader = ML.create_loader(test_dataset, batch_size=64, shuffle = False)
ML.visualize_loader(train_loader,max_images=10,nrow=5)


In [ ]:
import torch
import matplotlib.pyplot as plt
import wandb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = GlyphClassifier(resolution=(128, 128)).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)



# Training loop
num_epochs = 10
losses = []

#Initialising Weights&Biasis

wandb.init(
    project="glyph-regression",
    name="exp-1024neuron-128x128res-RandomStarsL",
    config={
        "architecture": "CNN-Glyph",
        "epochs": num_epochs,
        "batch_size": 64,
        "learning_rate": 0.0005,
        "loss_fn": "SmoothL1Loss",
        "optimizer": "Adam",
        "image_resolution": (128, 128),
        "regression": True
    }
)

# Watch model with Weights & Biases 
wandb.watch(model, log="all", log_freq=10)

for epoch in range(num_epochs):
    model.train()
    for images, labels, _ in train_loader:
        images = images.to(device)
        labels = labels.to(device)\
        
        optimizer.zero_grad()
        outputs = model(images).squeeze()
        loss = criterion(outputs, labels.squeeze())
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        

    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {losses[-1]:.4f}")
    wandb.log({"epoch": epoch + 1, "train_loss": losses[-1]})


    
ML.plot_training_loss(losses)

In [ ]:
import numpy as np 

model.eval()
predictions = []
ground_truths = []

with torch.no_grad():
    for images, labels, _ in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images).squeeze()
        predictions.extend(outputs.cpu().numpy())
        ground_truths.extend(labels.cpu().numpy())

predictions = np.array(predictions)
ground_truths = np.array(ground_truths)

mse = np.mean((predictions - ground_truths)**2)
mae = np.mean(np.abs(predictions - ground_truths))

print(f"Regression MSE: {mse:.4f}")
print(f"Regression MAE: {mae:.4f}")
wandb.log({
    "test_mse": mse,
    "test_mae": mae
})

wandb.finish()

## Binned Regression

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import src.machine_learning as ML  

class GlyphClassifier(nn.Module):
    def __init__(self, resolution, num_bins=20):
        super(GlyphClassifier, self).__init__()
        self.num_bins = num_bins
        
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        flattened_dim = 128 * resolution[0] // 8 * resolution[1] // 8
        self.classifier = nn.Sequential(
            nn.Linear(flattened_dim, 128),
            nn.ReLU(),
            nn.Linear(128, self.num_bins)  # Outputs logits for bins
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        logits = self.classifier(x)
        probs = F.softmax(logits, dim=1)  # Probabilities over bins

        bin_centers = torch.linspace(0, 1, steps=self.num_bins).to(x.device)
        expected_value = torch.sum(probs * bin_centers, dim=1, keepdim=True)  # Weighted avg

        return expected_value  # Shape: (B, 1)


In [ ]:
# Getting the dataset
dataset_file = 'data/simple-star.zip'
train_dataset = ML.GlyphDataset(dataset_file, resize=(128,128), split = "train",mode="regression")
test_dataset = ML.GlyphDataset(dataset_file, resize=(128,128),split = 'test',mode='regression')
train_dataset.show()

# Assign the loaders 

train_loader = ML.create_loader(train_dataset, batch_size=64, shuffle = True)
test_loader = ML.create_loader(test_dataset, batch_size=64, shuffle = False)
ML.visualize_loader(train_loader,max_images=10,nrow=5)


In [ ]:
import torch
import matplotlib.pyplot as plt
import wandb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = GlyphClassifier(resolution=(128, 128), num_bins=20).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)



# Training loop
num_epochs = 10
losses = []

#Initialising Weights&Biasis

wandb.init(
    project="glyph-regression",
    name="exp-SimpleStar-128neuron-128x128res-20bins-ActualScale",
    config={
        "architecture": "CNN-Glyph",
        "epochs": num_epochs,
        "batch_size": 64,
        "learning_rate": 0.0005,
        "loss_fn": "SmoothL1Loss",
        "optimizer": "Adam",
        "image_resolution": (128, 128),
        "regression": True
    }
)

# Watch model with Weights & Biases 
wandb.watch(model, log="all", log_freq=10)

for epoch in range(num_epochs):
    model.train()
    epoch_losses = []
    all_outputs = []
    all_labels = []

    for images, labels, _ in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images).squeeze()
        loss = criterion(outputs, labels.squeeze())
        loss.backward()
        optimizer.step()

        epoch_losses.append(loss.item())
        
        # Store actual values for MAE in real scale
        all_outputs.append(outputs.detach().cpu() * 100.0)
        all_labels.append(labels.cpu() * 100.0)

    # Combine all batches and compute real-scale MAE
    all_outputs = torch.cat(all_outputs)
    all_labels = torch.cat(all_labels)
    mae_actual = F.l1_loss(all_outputs, all_labels).item()

    mean_loss = sum(epoch_losses) / len(epoch_losses)
    losses.append(mean_loss)

    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {mean_loss:.4f} - MAE (Actual): {mae_actual:.2f}")
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": mean_loss,
        "train_mae_actual": mae_actual
    })


    
ML.plot_training_loss(losses)

In [ ]:
import numpy as np 

model.eval()
predictions = []
ground_truths = []

with torch.no_grad():
    for images, labels, _ in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images).squeeze()
        predictions.extend(outputs.cpu().numpy())
        ground_truths.extend(labels.cpu().numpy())

predictions = np.array(predictions) * 100.0
ground_truths = np.array(ground_truths) * 100.0

mse = np.mean((predictions - ground_truths)**2)
mae = np.mean(np.abs(predictions - ground_truths))

print(f"Regression MSE: {mse:.4f}")
print(f"Regression MAE: {mae:.4f}")
wandb.log({
    "test_mse": mse,
    "test_mae": mae
})

wandb.finish()